In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from transformers import (
    BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
)
from torch.utils.data import Dataset
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
# Load data
jailbreak_df = pd.read_csv('/content/all_labelled_cleaned.csv')
regular_df = pd.read_csv('/content/regular_prompts_DAN.csv')
# Preprocessing setup
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

In [ ]:
def preprocess_text(text):
    words = text.split()
    return ' '.join([stemmer.stem(word) for word in words if word.lower() not in stop_words])

def create_dataset(df, regular_df, label_name, train=True):
    if train:
        subset_df = df[~df['Labels'].apply(lambda labels: label_name in labels)].copy()
    else:
        subset_df = df[df['Labels'].apply(lambda labels: label_name in labels)].copy()

    subset_df['Jailbreak'] = 1
    subset_df = subset_df[['prompt', 'Jailbreak']]

    sampled_regular = regular_df.sample(n=len(subset_df), random_state=1).reset_index(drop=True)
    sampled_regular['Jailbreak'] = 0
    sampled_regular['prompt'] = sampled_regular['prompt'].apply(preprocess_text)
    sampled_regular = sampled_regular[['prompt', 'Jailbreak']]

    combined_df = pd.concat([subset_df, sampled_regular], ignore_index=True)
    return combined_df.sample(frac=1, random_state=1).reset_index(drop=True)


In [ ]:
# Create train/test datasets for each label
labels = ['Character Roleplay', 'Superior Model', 'Sudo Mode', 'Simulate Jailbreaking', 'Ethical Appeal']
train_dfs, test_dfs = [], []
for label in labels:
    train_dfs.append(create_dataset(jailbreak_df, regular_df, label, train=True))
    test_dfs.append(create_dataset(jailbreak_df, regular_df, label, train=False))

# Dataset class
class PromptDataset(Dataset):
    def __init__(self, prompts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(prompts, truncation=True, padding=True, max_length=max_length)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
            'labels': torch.tensor(self.labels[idx])
        }

# Compute metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fnr = fn / (tp + fn) if (tp + fn) > 0 else 0
    return {'accuracy': acc, 'f1': f1, 'tpr': tpr, 'fnr': fnr}


In [ ]:
# Training loop
os.environ["WANDB_MODE"] = "disabled"
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

metrics_dict = {}

for i, (train_df, test_df) in enumerate(zip(train_dfs, test_dfs)):
    train_dataset = PromptDataset(train_df['prompt'].tolist(), train_df['Jailbreak'].tolist(), tokenizer)
    test_dataset = PromptDataset(test_df['prompt'].tolist(), test_df['Jailbreak'].tolist(), tokenizer)

    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
    model_output_dir = f'./saved_model_{i}'
    os.makedirs(model_output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=model_output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=10,
        weight_decay=0.01,
        logging_dir=os.path.join(model_output_dir, 'logs'),
        logging_steps=10,
        eval_strategy="epoch"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    trainer.save_model(model_output_dir)
    tokenizer.save_pretrained(model_output_dir)

    eval_results = trainer.evaluate()
    metrics_dict[f'Model_{i+1}'] = eval_results

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Tpr,Fnr
1,0.366900,0.164113,0.976136,0.976190,0.978409,0.021591
2,0.008700,0.076831,0.981250,0.980980,0.967045,0.032955
3,0.001900,0.094092,0.977273,0.977350,0.980682,0.019318


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Tpr,Fnr
1,0.218200,0.005053,1.000000,1.000000,1.000000,0.000000
2,0.003200,0.000406,1.000000,1.000000,1.000000,0.000000
3,0.000200,0.000126,1.000000,1.000000,1.000000,0.000000


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Tpr,Fnr
1,0.078700,0.002743,1.000000,1.000000,1.000000,0.000000
2,0.060500,0.000385,1.000000,1.000000,1.000000,0.000000
3,0.066200,0.000167,1.000000,1.000000,1.000000,0.000000


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Tpr,Fnr
1,0.331600,0.196099,0.960784,0.960000,0.941176,0.058824
2,0.049800,0.001986,1.000000,1.000000,1.000000,0.000000
3,0.054700,0.141163,0.970588,0.969697,0.941176,0.058824


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Tpr,Fnr
1,0.012000,0.053923,0.980000,0.979592,0.960000,0.040000
2,0.058600,0.134628,0.960000,0.958333,0.920000,0.080000
3,0.000300,0.177495,0.960000,0.958333,0.920000,0.080000
